# Phase 5 — Business Metrics & Uncertainty-Gating Ablation Study

This notebook evaluates the **Recovery Decision Engine** on 500 held-out evaluation events (`data/eval.csv`) to compute financial and operational metrics, and performs an **Ablation Study** proving why Bayesian uncertainty gating is essential.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.evaluation.business_eval import run_ablation_study, run_batch_evaluation

sns.set_theme(style="whitegrid")
eval_df = pd.read_csv("../data/eval.csv")
print(f"Held-out Evaluation Dataset: {len(eval_df)} events")

## 1. Business Metrics & Ablation Results

In [ ]:
results = run_ablation_study(eval_df, model_path="../data/pymc_model_idata.pkl")
full_metrics = results["full_engine"]
ablated_metrics = results["ablated_engine"]

summary_df = pd.DataFrame([
    {
        "Configuration": "Full Engine (Uncertainty-Gated, tau=0.25)",
        "Rupees Recovered": f"₹{full_metrics['rupees_recovered']:,.2f}",
        "Rupees Wasted": f"₹{full_metrics['rupees_wasted']:,.2f}",
        "Rupees Avoided": f"₹{full_metrics['rupees_avoided']:,.2f}",
        "Automation Rate": f"{full_metrics['automation_rate']*100:.1f}%",
        "Escalation Rate": f"{full_metrics['escalation_rate']*100:.1f}%",
        "False Intervention Rate": f"{full_metrics['false_intervention_rate']*100:.1f}%",
    },
    {
        "Configuration": "Ablated Engine (Point-Estimate Only, tau=infinity)",
        "Rupees Recovered": f"₹{ablated_metrics['rupees_recovered']:,.2f}",
        "Rupees Wasted": f"₹{ablated_metrics['rupees_wasted']:,.2f}",
        "Rupees Avoided": f"₹{ablated_metrics['rupees_avoided']:,.2f}",
        "Automation Rate": f"{ablated_metrics['automation_rate']*100:.1f}%",
        "Escalation Rate": f"{ablated_metrics['escalation_rate']*100:.1f}%",
        "False Intervention Rate": f"{ablated_metrics['false_intervention_rate']*100:.1f}%",
    }
])

display(summary_df)

## 2. False Intervention Rate Comparison Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
configs = ["Full Uncertainty-Gated Engine", "Ablated (Point-Estimate Only)"]
rates = [full_metrics['false_intervention_rate'] * 100, ablated_metrics['false_intervention_rate'] * 100]
colors = ["#2ca02c", "#d62728"]

bars = ax.bar(configs, rates, color=colors, width=0.4)
ax.set_ylabel("False Intervention Rate on Hard Declines (%)", fontsize=12)
ax.set_title("Ablation Proof: Impact of Uncertainty Gating on Hard Declines", fontsize=14, fontweight="bold")
ax.set_ylim(0, max(rates) * 1.3)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f"{yval:.1f}%", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("../data/ablation_false_interventions.png", dpi=150)
plt.show()